In [ ]:
import os, sys, base64, json
sys.path.append('..')
from dotenv import load_dotenv
import anthropic
from utils.helpers import CLAUDE_SONNET, CLAUDE_HAIKU, CLAUDE_OPUS, DEFAULT_MODEL

load_dotenv()
client = anthropic.Anthropic()
MODEL = DEFAULT_MODEL


## 1. Extended Thinking

Extended Thinking permite a Claude "pensar" antes de responder, útil para problemas complejos.

In [ ]:
if "client" not in dir():
    import sys; sys.path.append('..')
    from dotenv import load_dotenv
    import anthropic
    from utils.helpers import DEFAULT_MODEL
    load_dotenv()
    client = anthropic.Anthropic()
    MODEL = DEFAULT_MODEL

# Extended Thinking requiere claude-sonnet-4-5 o superior
MODEL_THINKING = DEFAULT_MODEL

problema_logico = """
En una carrera, si María está detrás de Juan, y Pedro está delante de María 
pero detrás de Ana, y Carlos está entre Pedro y Juan, ¿en qué posición 
terminó cada uno? Ordénalos del primero al último.
"""

# Con Extended Thinking
response = client.messages.create(
    model=MODEL_THINKING,
    max_tokens=8000,  # Debe ser mayor que budget_tokens
    thinking={
        "type": "enabled",
        "budget_tokens": 5000  # Tokens para el pensamiento interno
    },
    messages=[{"role": "user", "content": problema_logico}]
)

print("=== Extended Thinking ===")
for block in response.content:
    if block.type == "thinking":
        print(f"\nPensamiento interno ({len(block.thinking)} chars):")
        print(block.thinking[:500] + "...")
    elif block.type == "text":
        print(f"\nRespuesta final:")
        print(block.text)


## 2. Image Support

In [ ]:
# Claude puede analizar imágenes de dos formas:
# 1. Base64 (imagen local)
# 2. URL (imagen en internet)

# Análisis de imagen desde URL
response = client.messages.create(
    model=MODEL,
    max_tokens=500,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "image",
                "source": {
                    "type": "url",
                    "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"
                }
            },
            {
                "type": "text",
                "text": "Describe esta imagen en detalle. ¿Qué elementos puedes identificar?"
            }
        ]
    }]
)

print("Descripción de imagen:")
print(response.content[0].text)

In [ ]:
# Imagen desde base64 (útil para imágenes locales)
def image_to_base64(image_path: str) -> tuple[str, str]:
    """Convierte imagen local a base64. Retorna (base64_str, media_type)."""
    import imghdr
    with open(image_path, "rb") as f:
        image_data = f.read()
    
    b64 = base64.standard_b64encode(image_data).decode("utf-8")
    ext = imghdr.what(None, image_data) or "jpeg"
    media_type = f"image/{ext}"
    return b64, media_type

def analyze_image_b64(image_path: str, question: str) -> str:
    """Analiza una imagen local con Claude."""
    b64, media_type = image_to_base64(image_path)
    
    response = client.messages.create(
        model=MODEL, max_tokens=500,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {
                    "type": "base64",
                    "media_type": media_type,
                    "data": b64
                }},
                {"type": "text", "text": question}
            ]
        }]
    )
    return response.content[0].text

# Para usar con imagen local:
# result = analyze_image_b64("mi_imagen.png", "¿Qué muestra esta imagen?")
print("✅ Función analyze_image_b64 lista para usar con imágenes locales")

## 3. Citations

In [ ]:
# Citations permite que Claude cite exactamente las partes del documento
# que soportan cada afirmación en su respuesta

documento_ciencia = """
La fotosíntesis es el proceso mediante el cual las plantas convierten 
la luz solar en energía química. Este proceso ocurre principalmente 
en los cloroplastos, específicamente en la clorofila. 
La ecuación general es: 6CO2 + 6H2O + luz → C6H12O6 + 6O2.
La fotosíntesis tiene dos etapas: las reacciones de luz y el ciclo de Calvin.
Las plantas producen aproximadamente 260 gigatoneladas de biomasa por año.
"""

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "document",
                "source": {
                    "type": "text",
                    "media_type": "text/plain",
                    "data": documento_ciencia
                },
                "title": "Manual de Biología - Fotosíntesis",
                "citations": {"enabled": True}
            },
            {
                "type": "text",
                "text": "¿Dónde ocurre la fotosíntesis y cuál es su ecuación química?"
            }
        ]
    }]
)

print("Respuesta con citas:")
for block in response.content:
    if block.type == "text":
        print(block.text)
    elif block.type == "citations":
        print("\n📚 Citas:")
        for cita in block.citations:
            print(f"  '{cita.cited_text}' (doc: {cita.document_title})")

## 4. Prompt Caching

In [ ]:
import time

# El prompt caching guarda partes del prompt en cache
# Ahorra hasta 90% en costo y reduce latencia
# Se marca con cache_control: {"type": "ephemeral"}

# Documento largo que se reutilizará
documento_largo = """
MANUAL COMPLETO DE PYTHON - VERSIÓN 3.12

CAPÍTULO 1: INTRODUCCIÓN
Python es un lenguaje de programación de alto nivel, interpretado y de propósito general.
Fue creado por Guido van Rossum y lanzado por primera vez en 1991.
Su filosofía de diseño enfatiza la legibilidad del código.

CAPÍTULO 2: TIPOS DE DATOS
Python tiene varios tipos de datos integrados: int, float, str, bool, list, tuple, dict, set.
Los strings son inmutables y soportan indexación y slicing.
Las listas son mutables y pueden contener elementos de diferentes tipos.

CAPÍTULO 3: CONTROL DE FLUJO
Las estructuras de control incluyen: if/elif/else, for, while, try/except.
Python usa indentación (4 espacios) en lugar de llaves para definir bloques.
""" * 5  # Repetir para simular documento largo

# Primera llamada - crea el cache
t0 = time.time()
resp1 = client.messages.create(
    model=MODEL,
    max_tokens=200,
    system=[
        {
            "type": "text",
            "text": "Eres un experto en Python. Responde preguntas basándote en el manual:"
        },
        {
            "type": "text",
            "text": documento_largo,
            "cache_control": {"type": "ephemeral"}  # ← Marcar para cache
        }
    ],
    messages=[{"role": "user", "content": "¿En qué año fue creado Python?"}]
)
t1 = time.time() - t0

print(f"Primera llamada: {t1:.2f}s")
print(f"  cache_creation_input_tokens: {resp1.usage.cache_creation_input_tokens}")
print(f"  cache_read_input_tokens:     {resp1.usage.cache_read_input_tokens}")
print(f"  Respuesta: {resp1.content[0].text.strip()[:100]}")

# Segunda llamada - usa el cache
t0 = time.time()
resp2 = client.messages.create(
    model=MODEL,
    max_tokens=200,
    system=[
        {"type": "text", "text": "Eres un experto en Python. Responde preguntas basándote en el manual:"},
        {"type": "text", "text": documento_largo, "cache_control": {"type": "ephemeral"}}
    ],
    messages=[{"role": "user", "content": "¿Qué tipos de datos tiene Python?"}]
)
t2 = time.time() - t0

print(f"\nSegunda llamada (con cache): {t2:.2f}s")
print(f"  cache_creation_input_tokens: {resp2.usage.cache_creation_input_tokens}")
print(f"  cache_read_input_tokens:     {resp2.usage.cache_read_input_tokens}")
print(f"  Ahorro: {((t1-t2)/t1)*100:.0f}% más rápido" if t1 > t2 else "  (Cache en proceso)")